# Semantic + Thematic Music Recommendation System - Full Code Notebook

This notebook contains the full project source code for review/submission. The production app still runs from the Python files using `streamlit run app.py`.

## 0. Setup

In [3]:
# Run once if dependencies are missing:
# !pip install -r requirements.txt

import os
os.chdir('/Users/suvratsmacbookair/MyLocalFiles/LyricLens')
print(os.getcwd())


/Users/suvratsmacbookair/MyLocalFiles/LyricLens


## Full Source: `preprocess.py`

In [ ]:
"""Data cleaning utilities for the music recommender."""

from __future__ import annotations

import reu?u?
from typing import Iterable

import pandas as pd


REQUIRED_COLUMNS = ["song_name", "artist", "lyrics", "genre"]


GENRE_ALIASES = {
    "hip-hop": ["hip hop", "hip-hop", "rap", "trap", "boom bap"],
    "rock": ["rock", "alt rock", "alternative rock", "classic rock", "hard rock", "punk"],
    "metal": ["metal", "heavy metal", "death metal", "black metal", "nu metal", "metalcore"],
    "pop": ["pop", "dance pop", "electropop", "synthpop"],
    "country": ["country"],
    "electronic": ["electronic", "edm", "house", "techno", "trance", "dubstep"],
    "folk": ["folk", "singer songwriter", "acoustic"],
    "r&b": ["r&b", "rnb", "soul"],
    "blues": ["blues"],
    "jazz": ["jazz"],
    "hindi": ["hindi", "bollywood"],
    "punjabi": ["punjabi"],
}


def validate_columns(df: pd.DataFrame, required_columns: Iterable[str] = REQUIRED_COLUMNS) -> None:
    """Raise a clear error if the dataset is missing required columns."""
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise ValueError(f"Dataset is missing required columns: {missing}")


def normalize_genre(value: object) -> str:
    """Map noisy genre strings into stable display buckets."""
    if pd.isna(value):
        return "unknown"

    text = str(value).encode("utf-8", errors="ignore").decode("utf-8", errors="ignore")
    text = text.lower().strip()
    if not text or text in {"nan", "none", "[]"}:
        return "unknown"

    text = re.sub(r"[\[\]\"']", " ", text)
    text = re.sub(r"[_/]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    for canonical, aliases in GENRE_ALIASES.items():
        if any(re.search(rf"\b{re.escape(alias)}\b", text) for alias in aliases):
            return canonical

    return text.split(",")[0].strip() or "unknown"


def clean_lyrics(text: object) -> str:
    """Lowercase lyrics, remove markup/noise/special characters, and normalize spaces."""
    if pd.isna(text):
        return ""

    text = str(text).encode("utf-8", errors="ignore").decode("utf-8", errors="ignore")
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\[[^\]]+\]|\([^\)]*(chorus|verse|bridge|intro|outro)[^\)]*\)", " ", text)
    text = re.sub(r"\b(embed|lyrics|contributors|translations?)\b", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\b(\w+)(\s+\1\b){3,}", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def word_count(text: str) -> int:
    """Count whitespace-delimited words after cleaning."""
    return len(text.split())


def preprocess_dataframe(df: pd.DataFrame, min_words: int = 20) -> pd.DataFrame:
    """Clean the input dataset and remove rows that are too short for embeddings."""
    validate_columns(df)

    cleaned = df.copy()
    cleaned["song_name"] = (
        cleaned["song_name"]
        .fillna("")
        .apply(lambda value: str(value).encode("utf-8", errors="ignore").decode("utf-8", errors="ignore").strip())
    )
    cleaned["artist"] = (
        cleaned["artist"]
        .fillna("")
        .apply(lambda value: str(value).encode("utf-8", errors="ignore").decode("utf-8", errors="ignore").strip())
    )
    cleaned["genre"] = cleaned["genre"].apply(normalize_genre)
    cleaned["lyrics"] = cleaned["lyrics"].fillna("").astype(str)

    cleaned = cleaned[
        (cleaned["song_name"] != "")
        & (cleaned["artist"] != "")
        & (cleaned["genre"] != "")
        & (cleaned["genre"] != "unknown")
        & (cleaned["lyrics"] != "")
    ].copy()

    cleaned["clean_lyrics"] = cleaned["lyrics"].apply(clean_lyrics)
    cleaned["word_count"] = cleaned["clean_lyrics"].apply(word_count)
    cleaned = cleaned[cleaned["word_count"] >= min_words].copy()
    cleaned["lyrics_hash"] = pd.util.hash_pandas_object(cleaned["clean_lyrics"], index=False).astype(str)

    cleaned = cleaned.drop_duplicates(subset=["song_name", "artist"], keep="first")
    cleaned = cleaned.drop_duplicates(subset=["lyrics_hash"], keep="first")
    cleaned = cleaned.reset_index(drop=True)
    return cleaned


## Full Source: `emotion_theme.py`

In [ ]:
"""Explainable emotion and theme tagging.

This module uses weighted lexicons, phrase matching, and related-label
similarity. It is still lightweight enough for a college prototype, but it is
more robust than simple one-keyword matching.
"""

from __future__ import annotations

import math
import re
from typing import Dict, Iterable, Mapping

import pandas as pd

from preprocess import clean_lyrics


EMOTION_LEXICON: Dict[str, Dict[str, float]] = {
    "happy": {
        "happy": 2.0,
        "joy": 2.0,
        "smile": 1.6,
        "dance": 1.2,
        "sunshine": 1.5,
        "celebrate": 1.7,
        "laugh": 1.4,
        "bright": 1.1,
        "free": 1.0,
        "glad": 1.3,
        "alive": 1.0,
    },
    "sad": {
        "sad": 2.0,
        "tears": 2.0,
        "cry": 1.8,
        "broken": 1.6,
        "empty": 1.7,
        "hollow": 1.7,
        "alone": 1.5,
        "miss": 1.3,
        "pain": 1.5,
        "blue": 1.2,
        "grief": 2.2,
        "hurt": 1.4,
        "dark": 0.9,
    },
    "angry": {
        "angry": 2.0,
        "rage": 2.1,
        "fire": 1.2,
        "fight": 1.2,
        "hate": 2.0,
        "storm": 1.0,
        "burn": 1.2,
        "shout": 1.3,
        "revenge": 2.0,
        "war": 1.4,
        "blood": 1.0,
    },
    "calm": {
        "calm": 2.0,
        "quiet": 1.8,
        "peace": 2.0,
        "soft": 1.2,
        "slow": 1.0,
        "breathe": 1.6,
        "still": 1.4,
        "gentle": 1.5,
        "river": 0.9,
        "moon": 0.9,
        "sleep": 1.0,
    },
    "motivational": {
        "rise": 1.8,
        "dream": 1.3,
        "win": 1.8,
        "hustle": 1.7,
        "strong": 1.6,
        "climb": 1.5,
        "build": 1.2,
        "goal": 1.7,
        "believe": 1.5,
        "champion": 1.8,
        "survive": 1.2,
    },
    "romantic": {
        "love": 2.0,
        "heart": 1.5,
        "kiss": 1.8,
        "darling": 1.6,
        "forever": 1.3,
        "hold": 1.0,
        "touch": 1.2,
        "beloved": 1.7,
        "together": 1.2,
        "romance": 2.0,
        "baby": 0.8,
    },
}


THEME_LEXICON: Dict[str, Dict[str, float]] = {
    "heartbreak": {
        "heartbreak": 2.4,
        "breakup": 2.3,
        "left me": 2.0,
        "goodbye": 1.8,
        "betray": 1.8,
        "broken heart": 2.2,
        "without you": 1.8,
        "farewell": 1.4,
        "door closed": 1.4,
    },
    "loneliness": {
        "alone": 1.8,
        "lonely": 2.2,
        "empty": 1.7,
        "hollow": 1.7,
        "silence": 1.6,
        "no one": 2.0,
        "distant": 1.2,
        "isolated": 2.0,
        "shadow": 1.2,
    },
    "loss/grief": {
        "death": 2.3,
        "lost": 1.7,
        "gone": 1.5,
        "miss you": 2.0,
        "grave": 2.1,
        "funeral": 2.3,
        "ashes": 1.8,
        "memory": 1.0,
        "grief": 2.4,
        "mourning": 2.2,
    },
    "guilt/regret": {
        "sorry": 1.8,
        "fault": 2.0,
        "regret": 2.3,
        "blame": 2.0,
        "forgive": 1.7,
        "mistake": 1.8,
        "apology": 2.0,
        "shame": 1.8,
        "my fault": 2.2,
        "confess": 1.4,
    },
    "self-reflection": {
        "mirror": 1.8,
        "inside": 1.2,
        "question": 1.3,
        "truth": 1.5,
        "myself": 1.8,
        "soul": 1.2,
        "thinking": 1.2,
        "learn": 1.2,
        "change": 1.2,
        "reflection": 2.0,
    },
    "ambition": {
        "ambition": 2.2,
        "dream": 1.5,
        "city": 0.8,
        "crown": 1.3,
        "goal": 1.8,
        "money": 1.2,
        "stage": 1.0,
        "legacy": 1.7,
        "future": 1.4,
        "grind": 1.5,
    },
    "struggle": {
        "struggle": 2.2,
        "battle": 1.8,
        "survive": 1.8,
        "pressure": 1.5,
        "scar": 1.5,
        "fight": 1.4,
        "pain": 1.2,
        "hard road": 1.8,
        "heavy": 0.9,
        "broken road": 1.7,
    },
}


EMOTION_RELATEDNESS: Mapping[tuple[str, str], float] = {
    ("sad", "calm"): 0.35,
    ("sad", "romantic"): 0.30,
    ("sad", "angry"): 0.25,
    ("angry", "motivational"): 0.30,
    ("happy", "romantic"): 0.45,
    ("happy", "motivational"): 0.35,
    ("calm", "romantic"): 0.30,
}


THEME_RELATEDNESS: Mapping[tuple[str, str], float] = {
    ("heartbreak", "loneliness"): 0.55,
    ("heartbreak", "loss/grief"): 0.45,
    ("heartbreak", "guilt/regret"): 0.35,
    ("loneliness", "loss/grief"): 0.35,
    ("loneliness", "self-reflection"): 0.35,
    ("loss/grief", "guilt/regret"): 0.30,
    ("guilt/regret", "self-reflection"): 0.45,
    ("struggle", "self-reflection"): 0.35,
    ("struggle", "ambition"): 0.45,
    ("ambition", "self-reflection"): 0.25,
}


def _weighted_hits(text: str, lexicon: Mapping[str, float]) -> float:
    score = 0.0
    for keyword, weight in lexicon.items():
        cleaned_keyword = clean_lyrics(keyword)
        if not cleaned_keyword:
            continue
        if " " in cleaned_keyword:
            score += text.count(cleaned_keyword) * weight
        else:
            score += len(re.findall(rf"\b{re.escape(cleaned_keyword)}\b", text)) * weight
    return score


def score_labels(text: object, lexicons: Dict[str, Dict[str, float]]) -> dict[str, float]:
    """Return normalized-ish scores for all labels."""
    cleaned = clean_lyrics(text)
    raw_scores = {
        label: _weighted_hits(cleaned, weighted_terms)
        for label, weighted_terms in lexicons.items()
    }
    word_count = max(1, len(cleaned.split()))
    return {label: score / math.sqrt(word_count) for label, score in raw_scores.items()}


def classify_with_confidence(
    text: object,
    lexicons: Dict[str, Dict[str, float]],
    default_label: str,
) -> tuple[str, float]:
    """Return best label and a confidence score between 0 and 1."""
    scores = score_labels(text, lexicons)
    ordered = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    best_label, best_score = ordered[0]
    second_score = ordered[1][1] if len(ordered) > 1 else 0.0

    if best_score <= 0:
        return default_label, 0.20

    confidence = min(0.99, 0.45 + best_score + max(0.0, best_score - second_score))
    return best_label, round(float(confidence), 3)


def detect_emotion(text: object) -> str:
    """Classify one of the required emotion labels."""
    return classify_with_confidence(text, EMOTION_LEXICON, default_label="calm")[0]


def detect_theme(text: object) -> str:
    """Classify one of the required theme labels."""
    return classify_with_confidence(text, THEME_LEXICON, default_label="self-reflection")[0]


def emotion_similarity(left: str, right: str) -> float:
    """Exact or related emotional-state similarity."""
    if left == right:
        return 1.0
    return float(EMOTION_RELATEDNESS.get((left, right), EMOTION_RELATEDNESS.get((right, left), 0.0)))


def theme_similarity(left: str, right: str) -> float:
    """Exact or related thematic similarity."""
    if left == right:
        return 1.0
    return float(THEME_RELATEDNESS.get((left, right), THEME_RELATEDNESS.get((right, left), 0.0)))


def add_emotion_theme_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Add labels and confidence scores to every song row."""
    labelled = df.copy()
    source_col = "clean_lyrics" if "clean_lyrics" in labelled.columns else "lyrics"

    emotion_results = labelled[source_col].apply(
        lambda text: classify_with_confidence(text, EMOTION_LEXICON, "calm")
    )
    theme_results = labelled[source_col].apply(
        lambda text: classify_with_confidence(text, THEME_LEXICON, "self-reflection")
    )

    labelled["emotion"] = emotion_results.apply(lambda item: item[0])
    labelled["emotion_confidence"] = emotion_results.apply(lambda item: item[1])
    labelled["theme"] = theme_results.apply(lambda item: item[0])
    labelled["theme_confidence"] = theme_results.apply(lambda item: item[1])
    return labelled


## Full Source: `data_loader.py`

In [ ]:
"""Multi-source dataset loading and alignment.

The app reads the user's downloaded Kaggle datasets from absolute paths,
samples them into a practical demo-sized corpus, and caches the unified CSV.
"""

from __future__ import annotations

import ast
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import pandas as pd

from emotion_theme import add_emotion_theme_labels
from preprocess import REQUIRED_COLUMNS, normalize_genre, preprocess_dataframe, validate_columns


ARTISTS_PATH = Path("/Users/suvratsmacbookair/Downloads/archive-3/artists.csv")
SPOTIFY_SONGS_PATH = Path("/Users/suvratsmacbookair/Downloads/archive-3/songs.csv")
CLEANED_TEST_LYRICS_PATH = Path("/Users/suvratsmacbookair/Downloads/archive-4/cleaned_test_lyrics.csv")
CLEANED_TRAIN_LYRICS_PATH = Path("/Users/suvratsmacbookair/Downloads/archive-4/cleaned_train_lyrics.csv")

UNIFIED_DATASET_PATH = Path("data/unified_songs.csv")
UNIFIED_METADATA_PATH = Path("data/unified_songs.meta.json")

CHUNK_SIZE = 75_000
MAX_ROWS_PER_SOURCE_GENRE = 90
FINAL_ROWS_PER_GENRE = 180


@dataclass(frozen=True)
class DatasetSource:
    name: str
    path: Path
    kind: str


DATASET_SOURCES = [
    DatasetSource("spotify_songs", SPOTIFY_SONGS_PATH, "spotify_songs"),
    DatasetSource("lyrics_test", CLEANED_TEST_LYRICS_PATH, "cleaned_lyrics"),
    DatasetSource("lyrics_train", CLEANED_TRAIN_LYRICS_PATH, "cleaned_lyrics"),
]


def _safe_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value).encode("utf-8", errors="ignore").decode("utf-8", errors="ignore")


def _require_paths(paths: Iterable[Path]) -> None:
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Required dataset files are missing:\n"
            + "\n".join(missing)
            + "\nDownload/extract the datasets to these exact paths, then run again."
        )


def _source_signature() -> dict[str, object]:
    paths = [ARTISTS_PATH, *(source.path for source in DATASET_SOURCES)]
    _require_paths(paths)
    return {
        "paths": {
            str(path): {
                "size": path.stat().st_size,
                "mtime_ns": path.stat().st_mtime_ns,
            }
            for path in paths
        },
        "chunk_size": CHUNK_SIZE,
        "max_rows_per_source_genre": MAX_ROWS_PER_SOURCE_GENRE,
        "final_rows_per_genre": FINAL_ROWS_PER_GENRE,
        "schema_version": 3,
    }


def _metadata_matches(signature: dict[str, object]) -> bool:
    if not UNIFIED_DATASET_PATH.exists() or not UNIFIED_METADATA_PATH.exists():
        return False
    try:
        existing = json.loads(UNIFIED_METADATA_PATH.read_text())
    except json.JSONDecodeError:
        return False
    return existing == signature


def _write_metadata(signature: dict[str, object]) -> None:
    UNIFIED_METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    UNIFIED_METADATA_PATH.write_text(json.dumps(signature, indent=2))


def _safe_literal_list(value: object) -> list[str]:
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    text = _safe_text(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return [text]
    if isinstance(parsed, list):
        return [_safe_text(item) for item in parsed]
    return [_safe_text(parsed)]


def _first_artist(value: object) -> str:
    artists = _safe_literal_list(value)
    return artists[0] if artists else "Unknown Artist"


def _first_artist_id(value: object) -> str:
    artist_ids = _safe_literal_list(value)
    return artist_ids[0] if artist_ids else ""


def load_artist_lookup(path: Path = ARTISTS_PATH) -> pd.DataFrame:
    """Load artist metadata and normalize it for optional genre enrichment."""
    artists = pd.read_csv(
        path,
        usecols=["id", "name", "main_genre", "genres"],
        encoding_errors="ignore",
    )
    artists["artist_main_genre"] = artists["main_genre"].apply(normalize_genre)
    artists = artists.rename(columns={"id": "artist_id", "name": "artist_lookup_name"})
    return artists[["artist_id", "artist_lookup_name", "artist_main_genre", "genres"]]


def _sample_by_genre(df: pd.DataFrame, max_per_genre: int, random_state: int) -> pd.DataFrame:
    if df.empty:
        return df

    sampled = []
    for genre, group in df.groupby("genre", sort=True):
        if len(group) > max_per_genre:
            sampled.append(group.sample(max_per_genre, random_state=random_state))
        else:
            sampled.append(group)
    return pd.concat(sampled, ignore_index=True) if sampled else df.head(0)


def _load_spotify_songs(source: DatasetSource, artist_lookup: pd.DataFrame) -> pd.DataFrame:
    """Read the large Spotify song CSV in chunks and sample per genre."""
    usecols = ["id", "name", "artists", "lyrics", "genre", "artist_ids", "niche_genres"]
    chunks: list[pd.DataFrame] = []
    running_counts: dict[str, int] = {}

    for chunk_id, chunk in enumerate(
        pd.read_csv(
            source.path,
            usecols=usecols,
            chunksize=CHUNK_SIZE,
            encoding_errors="ignore",
        )
    ):
        chunk = chunk.rename(columns={"name": "song_name"})
        chunk["artist"] = chunk["artists"].apply(_first_artist)
        chunk["artist_id"] = chunk["artist_ids"].apply(_first_artist_id)
        chunk["genre"] = chunk["genre"].apply(normalize_genre)
        chunk = chunk.merge(artist_lookup, how="left", on="artist_id")
        missing_genre = chunk["genre"].isin(["", "unknown"])
        chunk.loc[missing_genre, "genre"] = chunk.loc[missing_genre, "artist_main_genre"].fillna("unknown")

        aligned = pd.DataFrame(
            {
                "song_name": chunk["song_name"],
                "artist": chunk["artist"].fillna(chunk["artist_lookup_name"]).fillna("Unknown Artist"),
                "lyrics": chunk["lyrics"],
                "genre": chunk["genre"],
                "source": source.name,
            }
        )
        aligned = preprocess_dataframe(aligned, min_words=20)

        keep_parts = []
        for genre, group in aligned.groupby("genre", sort=False):
            already = running_counts.get(genre, 0)
            remaining = MAX_ROWS_PER_SOURCE_GENRE - already
            if remaining <= 0:
                continue
            selected = group.head(remaining)
            running_counts[genre] = already + len(selected)
            keep_parts.append(selected)

        if keep_parts:
            chunks.append(pd.concat(keep_parts, ignore_index=True))

        known_genres = {"rock", "metal", "hip-hop", "pop", "country", "electronic", "folk", "r&b", "blues", "jazz"}
        if known_genres.issubset({genre for genre, count in running_counts.items() if count >= MAX_ROWS_PER_SOURCE_GENRE}):
            break

    if not chunks:
        return pd.DataFrame(columns=[*REQUIRED_COLUMNS, "source", "clean_lyrics", "word_count"])
    return pd.concat(chunks, ignore_index=True)


def _load_cleaned_lyrics(source: DatasetSource) -> pd.DataFrame:
    """Read lyric classification CSVs that only contain lyrics and genre."""
    chunks: list[pd.DataFrame] = []
    running_counts: dict[str, int] = {}

    for chunk_id, chunk in enumerate(
        pd.read_csv(
            source.path,
            usecols=["Unnamed: 0", "Lyric", "genre"],
            chunksize=CHUNK_SIZE,
            encoding_errors="ignore",
        )
    ):
        chunk = chunk.rename(columns={"Lyric": "lyrics", "Unnamed: 0": "source_row"})
        chunk["genre"] = chunk["genre"].apply(normalize_genre)
        chunk["song_name"] = chunk.apply(
            lambda row: f"{source.name.replace('_', ' ').title()} Song {int(row['source_row'])}",
            axis=1,
        )
        chunk["artist"] = source.name.replace("_", " ").title()
        chunk["source"] = source.name

        aligned = preprocess_dataframe(chunk[[*REQUIRED_COLUMNS, "source"]], min_words=20)
        keep_parts = []
        for genre, group in aligned.groupby("genre", sort=False):
            already = running_counts.get(genre, 0)
            remaining = MAX_ROWS_PER_SOURCE_GENRE - already
            if remaining <= 0:
                continue
            selected = group.head(remaining)
            running_counts[genre] = already + len(selected)
            keep_parts.append(selected)

        if keep_parts:
            chunks.append(pd.concat(keep_parts, ignore_index=True))

        known_genres = {"metal", "rock", "hip-hop", "pop", "country"}
        if known_genres.issubset({genre for genre, count in running_counts.items() if count >= MAX_ROWS_PER_SOURCE_GENRE}):
            break

    if not chunks:
        return pd.DataFrame(columns=[*REQUIRED_COLUMNS, "source", "clean_lyrics", "word_count"])
    return pd.concat(chunks, ignore_index=True)


def build_unified_dataset() -> pd.DataFrame:
    """Load all configured external datasets, align schemas, sample, and tag."""
    signature = _source_signature()
    artist_lookup = load_artist_lookup()

    frames: list[pd.DataFrame] = []
    for source in DATASET_SOURCES:
        if source.kind == "spotify_songs":
            frames.append(_load_spotify_songs(source, artist_lookup))
        elif source.kind == "cleaned_lyrics":
            frames.append(_load_cleaned_lyrics(source))
        else:
            raise ValueError(f"Unknown dataset source kind: {source.kind}")

    merged = pd.concat(frames, ignore_index=True)
    merged = preprocess_dataframe(merged, min_words=20)
    merged = _sample_by_genre(merged, FINAL_ROWS_PER_GENRE, random_state=42)
    merged = add_emotion_theme_labels(merged)

    UNIFIED_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(UNIFIED_DATASET_PATH, index=False)
    _write_metadata(signature)
    return merged


def load_dataset(force_rebuild: bool = False) -> pd.DataFrame:
    """Load the cached unified dataset or rebuild it from all source CSVs."""
    signature = _source_signature()
    if not force_rebuild and _metadata_matches(signature):
        df = pd.read_csv(UNIFIED_DATASET_PATH)
        validate_columns(df)
        return df
    return build_unified_dataset()


def prepare_dataset(force_rebuild: bool = False) -> pd.DataFrame:
    """Load, clean, and ensure emotion/theme labels exist."""
    df = load_dataset(force_rebuild=force_rebuild)
    df = preprocess_dataframe(df, min_words=20)
    if "emotion" not in df.columns or "theme" not in df.columns:
        df = add_emotion_theme_labels(df)
    return df.reset_index(drop=True)


def dataset_summary(df: pd.DataFrame) -> dict[str, int]:
    """Return simple counts useful for UI and sanity checks."""
    return {
        "songs": int(len(df)),
        "genres": int(df["genre"].nunique()),
        "emotions": int(df["emotion"].nunique()),
        "themes": int(df["theme"].nunique()),
    }


## Full Source: `embedding.py`

In [ ]:
"""Sentence-BERT embedding utilities with efficient cache invalidation."""

from __future__ import annotations

import hashlib
import pickle
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd


PRIMARY_MODEL_NAME = "all-mpnet-base-v2"
FALLBACK_MODEL_NAME = "all-MiniLM-L6-v2"
MODEL_NAME = PRIMARY_MODEL_NAME
DEFAULT_EMBEDDING_PATH = Path("artifacts/lyrics_embeddings.pkl")


def _model_cache_name(model_name: str) -> str:
    return model_name.replace("/", "_").replace("-", "_")


def embedding_cache_path(model_name: str) -> Path:
    return Path("artifacts") / f"lyrics_embeddings_{_model_cache_name(model_name)}.pkl"


def load_embedding_model(
    model_name: str = PRIMARY_MODEL_NAME,
    fallback_model_name: str = FALLBACK_MODEL_NAME,
    local_files_only: bool = False,
):
    """Load mpnet when available, otherwise fall back to MiniLM."""
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as exc:
        raise ImportError(
            "sentence-transformers is not installed. Run: pip install -r requirements.txt"
        ) from exc

    attempts = [model_name]
    if fallback_model_name and fallback_model_name not in attempts:
        attempts.append(fallback_model_name)

    errors: list[str] = []
    for candidate in attempts:
        try:
            model = SentenceTransformer(candidate, local_files_only=local_files_only)
            setattr(model, "_selected_model_name", candidate)
            return model
        except Exception as exc:
            errors.append(f"{candidate}: {exc}")
            if not local_files_only:
                try:
                    model = SentenceTransformer(candidate, local_files_only=True)
                    setattr(model, "_selected_model_name", candidate)
                    return model
                except Exception as local_exc:
                    errors.append(f"{candidate} local cache: {local_exc}")

    raise RuntimeError(
        "Could not load a Sentence-BERT model. Connect to the internet once to "
        "download all-mpnet-base-v2 or all-MiniLM-L6-v2.\n" + "\n".join(errors[-3:])
    )


def selected_model_name(model) -> str:
    return str(getattr(model, "_selected_model_name", PRIMARY_MODEL_NAME))


def dataset_fingerprint(df: pd.DataFrame) -> str:
    """Create a stable fingerprint for cache invalidation."""
    fields = ["song_name", "artist", "genre", "clean_lyrics", "emotion", "theme"]
    available = [field for field in fields if field in df.columns]
    payload = df[available].astype(str).agg("||".join, axis=1).str.cat(sep="\n")
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def compute_embeddings(
    model,
    texts: Iterable[str],
    show_progress_bar: bool = False,
    batch_size: int = 32,
) -> np.ndarray:
    """Compute normalized dense embeddings for lyrics."""
    embeddings = model.encode(
        list(texts),
        batch_size=batch_size,
        show_progress_bar=show_progress_bar,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return np.asarray(embeddings, dtype=np.float32)


def load_or_create_embeddings(
    df: pd.DataFrame,
    model,
    cache_path: Path | str | None = None,
    force_rebuild: bool = False,
    show_progress_bar: bool = False,
) -> np.ndarray:
    """Load cached embeddings when possible, otherwise compute and persist them."""
    model_name = selected_model_name(model)
    cache_path = Path(cache_path) if cache_path else embedding_cache_path(model_name)
    fingerprint = dataset_fingerprint(df)

    if cache_path.exists() and not force_rebuild:
        with cache_path.open("rb") as file:
            cached = pickle.load(file)
        if (
            cached.get("model_name") == model_name
            and cached.get("fingerprint") == fingerprint
            and "embeddings" in cached
        ):
            return np.asarray(cached["embeddings"], dtype=np.float32)

    embeddings = compute_embeddings(model, df["clean_lyrics"].tolist(), show_progress_bar=show_progress_bar)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open("wb") as file:
        pickle.dump(
            {
                "model_name": model_name,
                "fingerprint": fingerprint,
                "embeddings": embeddings,
            },
            file,
        )
    return embeddings


def embed_single_text(model, text: str) -> np.ndarray:
    """Embed one custom text into the same normalized vector space."""
    embedding = model.encode([text], convert_to_numpy=True, normalize_embeddings=True)
    return np.asarray(embedding, dtype=np.float32)[0]


## Full Source: `recommender.py`

In [ ]:
"""Semantic + thematic music recommendation engine."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

from embedding import embed_single_text
from emotion_theme import detect_emotion, detect_theme, emotion_similarity, theme_similarity
from preprocess import clean_lyrics


SEMANTIC_WEIGHT = 0.62
EMOTION_WEIGHT = 0.14
THEME_WEIGHT = 0.14
CROSS_GENRE_WEIGHT = 0.10


@dataclass(frozen=True)
class QueryContext:
    song_name: str
    artist: str
    genre: str
    emotion: str
    theme: str
    lyrics_preview: str = ""


class MusicRecommender:
    """Score songs by semantics, emotional state, theme, and cross-genre contrast."""

    def __init__(self, df: pd.DataFrame, embeddings: np.ndarray, model: Any | None = None):
        if len(df) != len(embeddings):
            raise ValueError("DataFrame and embeddings must have the same number of rows.")

        self.df = df.reset_index(drop=True)
        self.embeddings = np.asarray(embeddings, dtype=np.float32)
        self.model = model

    def search_songs(self, query: str, limit: int = 50) -> pd.DataFrame:
        """Search songs by title or artist."""
        query = query.strip().lower()
        if not query:
            return self.df.head(limit)

        mask = (
            self.df["song_name"].str.lower().str.contains(query, regex=False)
            | self.df["artist"].str.lower().str.contains(query, regex=False)
        )
        return self.df[mask].head(limit)

    def find_song_index(self, song_name: str) -> int:
        """Find a song by exact case-insensitive name, falling back to contains matching."""
        query = song_name.strip().lower()
        exact = self.df.index[self.df["song_name"].str.lower() == query].tolist()
        if exact:
            return exact[0]

        contains = self.df.index[self.df["song_name"].str.lower().str.contains(query, regex=False)].tolist()
        if contains:
            return contains[0]

        raise ValueError(f"Song not found: {song_name}")

    def context_from_index(self, index: int) -> QueryContext:
        row = self.df.iloc[index]
        return QueryContext(
            song_name=row["song_name"],
            artist=row["artist"],
            genre=row["genre"],
            emotion=row["emotion"],
            theme=row["theme"],
            lyrics_preview=self.lyrics_preview(row),
        )

    def recommend_by_song(
        self,
        song_name: str,
        top_n: int = 5,
        candidate_indices: Optional[Iterable[int]] = None,
    ) -> pd.DataFrame:
        """Recommend songs similar to a selected catalog song."""
        index = self.find_song_index(song_name)
        return self.recommend_by_index(index, top_n=top_n, candidate_indices=candidate_indices)

    def recommend_by_index(
        self,
        index: int,
        top_n: int = 5,
        candidate_indices: Optional[Iterable[int]] = None,
    ) -> pd.DataFrame:
        """Recommend songs similar to a selected row index."""
        query = self.context_from_index(index)
        return self._rank(
            query_embedding=self.embeddings[index],
            query=query,
            top_n=top_n,
            exclude_index=index,
            candidate_indices=candidate_indices,
        )

    def recommend_by_custom_lyrics(
        self,
        lyrics: str,
        genre: str = "custom",
        top_n: int = 5,
        candidate_indices: Optional[Iterable[int]] = None,
    ) -> tuple[QueryContext, pd.DataFrame]:
        """Recommend songs for lyrics pasted by a user."""
        if self.model is None:
            raise ValueError("A SentenceTransformer model is required for custom lyrics recommendations.")

        cleaned = clean_lyrics(lyrics)
        if len(cleaned.split()) < 20:
            raise ValueError("Please enter at least 20 words so the semantic embedding has enough context.")

        query = QueryContext(
            song_name="Custom Lyrics",
            artist="User Input",
            genre=genre.lower().strip() or "custom",
            emotion=detect_emotion(cleaned),
            theme=detect_theme(cleaned),
            lyrics_preview=self.make_preview(cleaned),
        )
        query_embedding = embed_single_text(self.model, cleaned)
        recommendations = self._rank(
            query_embedding=query_embedding,
            query=query,
            top_n=top_n,
            exclude_index=None,
            candidate_indices=candidate_indices,
        )
        return query, recommendations

    def _rank(
        self,
        query_embedding: np.ndarray,
        query: QueryContext,
        top_n: int,
        exclude_index: Optional[int],
        candidate_indices: Optional[Iterable[int]],
    ) -> pd.DataFrame:
        semantic_scores = cosine_similarity([query_embedding], self.embeddings)[0]
        ranked = self.df.copy()
        ranked["semantic_similarity"] = semantic_scores
        ranked["emotion_similarity"] = ranked["emotion"].apply(lambda value: emotion_similarity(value, query.emotion))
        ranked["theme_similarity"] = ranked["theme"].apply(lambda value: theme_similarity(value, query.theme))
        ranked["genre_diff"] = (ranked["genre"] != query.genre).astype(int)
        ranked["final_score"] = (
            SEMANTIC_WEIGHT * ranked["semantic_similarity"]
            + EMOTION_WEIGHT * ranked["emotion_similarity"]
            + THEME_WEIGHT * ranked["theme_similarity"]
            + CROSS_GENRE_WEIGHT * ranked["genre_diff"]
        )

        if candidate_indices is not None:
            allowed = set(int(index) for index in candidate_indices)
            ranked = ranked[ranked.index.isin(allowed)].copy()

        if exclude_index is not None and exclude_index in ranked.index:
            ranked.loc[exclude_index, "final_score"] = -np.inf

        ranked = ranked.sort_values("final_score", ascending=False).head(top_n).copy()
        ranked["lyrics_preview"] = ranked.apply(self.lyrics_preview, axis=1)
        ranked["explanation"] = ranked.apply(lambda row: self._explain(query, row), axis=1)

        display_columns = [
            "song_name",
            "artist",
            "genre",
            "emotion",
            "theme",
            "semantic_similarity",
            "emotion_similarity",
            "theme_similarity",
            "genre_diff",
            "final_score",
            "lyrics_preview",
            "explanation",
        ]
        optional_columns = ["source", "word_count", "emotion_confidence", "theme_confidence"]
        return ranked[[*display_columns, *[col for col in optional_columns if col in ranked.columns]]].reset_index(drop=True)

    @staticmethod
    def make_preview(text: str, max_chars: int = 260) -> str:
        cleaned = " ".join(str(text).split())
        if len(cleaned) <= max_chars:
            return cleaned
        return cleaned[: max_chars - 3].rstrip() + "..."

    def lyrics_preview(self, row: pd.Series, max_chars: int = 260) -> str:
        source = row.get("lyrics", row.get("clean_lyrics", ""))
        return self.make_preview(str(source), max_chars=max_chars)

    @staticmethod
    def _explain(query: QueryContext, row: pd.Series) -> str:
        """Build a dynamic explanation for a recommendation."""
        cross_genre = row["genre"] != query.genre
        genre_line = (
            f"across genres ({query.genre} -> {row['genre']})"
            if cross_genre
            else f"within {row['genre']}"
        )

        if row["theme_similarity"] == 1.0:
            theme_line = f"same theme: {row['theme']}"
        elif row["theme_similarity"] > 0:
            theme_line = f"related theme: {query.theme} -> {row['theme']}"
        else:
            theme_line = f"semantic lyric similarity despite a different theme ({row['theme']})"

        if row["emotion_similarity"] == 1.0:
            emotion_line = f"same emotion: {row['emotion']}"
        elif row["emotion_similarity"] > 0:
            emotion_line = f"related emotion: {query.emotion} -> {row['emotion']}"
        else:
            emotion_line = f"different emotional color: {row['emotion']}"

        return (
            f"Recommended for high lyric-embedding similarity, {theme_line}, "
            f"{emotion_line}, {genre_line}."
        )


def filter_candidate_indices(
    df: pd.DataFrame,
    genres: Optional[list[str]] = None,
    emotions: Optional[list[str]] = None,
    themes: Optional[list[str]] = None,
) -> list[int]:
    """Return row indices matching UI filters."""
    mask = pd.Series(True, index=df.index)
    if genres:
        mask &= df["genre"].isin(genres)
    if emotions:
        mask &= df["emotion"].isin(emotions)
    if themes:
        mask &= df["theme"].isin(themes)
    return df.index[mask].tolist()


def recommendations_to_records(recommendations: pd.DataFrame) -> List[Dict[str, Any]]:
    """Small helper for notebooks, APIs, or tests."""
    return recommendations.to_dict(orient="records")


## Full Source: `app.py`

In [ ]:
"""Streamlit app for semantic + thematic music recommendations."""

from __future__ import annotations

import pandas as pd
import streamlit as st

from data_loader import dataset_summary, prepare_dataset
from embedding import load_embedding_model, load_or_create_embeddings, selected_model_name
from recommender import MusicRecommender, filter_candidate_indices


st.set_page_config(
    page_title="Semantic + Thematic Music Recommender",
    page_icon=":musical_note:",
    layout="wide",
)


st.markdown(
    """
    <style>
    .block-container {padding-top: 1.5rem; padding-bottom: 2rem;}
    div[data-testid="stMetric"] {
        background: #f8fafc;
        border: 1px solid #e5e7eb;
        border-radius: 8px;
        padding: 0.75rem;
    }
    .song-title {font-size: 1.05rem; font-weight: 700; margin-bottom: 0.1rem;}
    .tag {
        display: inline-block;
        padding: 0.16rem 0.45rem;
        border: 1px solid #d1d5db;
        border-radius: 999px;
        font-size: 0.78rem;
        color: #374151;
        background: #ffffff;
        margin-right: 0.25rem;
    }
    .cross-tag {border-color: #0f766e; color: #0f766e; background: #ecfdf5;}
    .lyrics-preview {
        color: #4b5563;
        font-size: 0.9rem;
        line-height: 1.35;
    }
    </style>
    """,
    unsafe_allow_html=True,
)


@st.cache_resource(show_spinner=False)
def load_system(force_rebuild_dataset: bool = False) -> tuple[MusicRecommender, str]:
    df = prepare_dataset(force_rebuild=force_rebuild_dataset)
    model = load_embedding_model()
    embeddings = load_or_create_embeddings(df, model, show_progress_bar=False)
    return MusicRecommender(df=df, embeddings=embeddings, model=model), selected_model_name(model)


def format_song_option(df: pd.DataFrame, index: int) -> str:
    row = df.loc[index]
    return f"{row['song_name']} - {row['artist']} [{row['genre']}]"


def score_to_percent(score: float) -> int:
    return max(0, min(100, int(round(float(score) * 100))))


def render_tags(row: pd.Series) -> None:
    st.markdown(
        f"""
        <span class="tag">{row['genre']}</span>
        <span class="tag">{row['emotion']}</span>
        <span class="tag">{row['theme']}</span>
        """,
        unsafe_allow_html=True,
    )


def render_selected_song(row: pd.Series, preview: str) -> None:
    with st.container(border=True):
        left, right = st.columns([0.72, 0.28])
        with left:
            st.markdown(f"<div class='song-title'>{row['song_name']}</div>", unsafe_allow_html=True)
            st.caption(f"{row['artist']} | source: {row.get('source', 'dataset')}")
            render_tags(row)
            st.markdown(f"<p class='lyrics-preview'>{preview}</p>", unsafe_allow_html=True)
        with right:
            st.metric("Words", int(row.get("word_count", 0)))
            st.metric("Emotion conf.", f"{float(row.get('emotion_confidence', 0.0)):.2f}")
            st.metric("Theme conf.", f"{float(row.get('theme_confidence', 0.0)):.2f}")


def render_recommendations(recommendations: pd.DataFrame) -> None:
    for rank, row in recommendations.iterrows():
        cross_genre = bool(row["genre_diff"])
        with st.container(border=True):
            top = st.columns([0.08, 0.48, 0.16, 0.14, 0.14])
            top[0].markdown(f"### #{rank + 1}")
            top[1].markdown(f"### {row['song_name']}")
            top[1].caption(row["artist"])
            top[2].metric("Final", f"{row['final_score']:.3f}")
            top[3].metric("Semantic", f"{row['semantic_similarity']:.3f}")
            top[4].metric("Theme", f"{row['theme_similarity']:.2f}")

            tag_class = "tag cross-tag" if cross_genre else "tag"
            tag_text = "Cross-genre match" if cross_genre else "Same-genre match"
            st.markdown(
                f"""
                <span class="{tag_class}">{tag_text}</span>
                <span class="tag">{row['genre']}</span>
                <span class="tag">{row['emotion']}</span>
                <span class="tag">{row['theme']}</span>
                """,
                unsafe_allow_html=True,
            )
            st.progress(score_to_percent(row["final_score"]))
            st.write(row["explanation"])
            with st.expander("Lyrics preview"):
                st.write(row["lyrics_preview"])


def render_analytics(df: pd.DataFrame, model_name: str) -> None:
    summary = dataset_summary(df)
    metric_cols = st.columns(5)
    metric_cols[0].metric("Songs", summary["songs"])
    metric_cols[1].metric("Genres", summary["genres"])
    metric_cols[2].metric("Emotions", summary["emotions"])
    metric_cols[3].metric("Themes", summary["themes"])
    metric_cols[4].metric("Embedding model", model_name)

    chart_cols = st.columns(3)
    chart_cols[0].subheader("Genres")
    chart_cols[0].bar_chart(df["genre"].value_counts())
    chart_cols[1].subheader("Emotions")
    chart_cols[1].bar_chart(df["emotion"].value_counts())
    chart_cols[2].subheader("Themes")
    chart_cols[2].bar_chart(df["theme"].value_counts())


def filtered_candidates(df: pd.DataFrame, genres: list[str], emotions: list[str], themes: list[str]) -> list[int]:
    return filter_candidate_indices(
        df,
        genres=genres if genres else None,
        emotions=emotions if emotions else None,
        themes=themes if themes else None,
    )


def main() -> None:
    st.title("Semantic + Thematic Music Recommendation System")
    st.caption("Sentence-BERT embeddings, emotion/theme inference, and cross-genre recommendation scoring")

    with st.spinner("Loading datasets, model, and cached embeddings..."):
        try:
            recommender, model_name = load_system(force_rebuild_dataset=False)
        except Exception as exc:
            st.error(str(exc))
            st.stop()

    df = recommender.df

    with st.sidebar:
        st.header("Controls")
        top_n = st.slider("Top-N recommendations", 3, 15, 5)
        search_query = st.text_input("Search song or artist")

        st.divider()
        st.subheader("Recommendation Filters")
        selected_genres = st.multiselect("Genre", sorted(df["genre"].dropna().unique()))
        selected_emotions = st.multiselect("Emotion", sorted(df["emotion"].dropna().unique()))
        selected_themes = st.multiselect("Theme", sorted(df["theme"].dropna().unique()))
        candidate_indices = filtered_candidates(df, selected_genres, selected_emotions, selected_themes)

        st.caption(f"{len(candidate_indices)} songs in filtered candidate pool")

    overview_tab, catalog_tab, custom_tab = st.tabs(["Overview", "Catalog Recommendation", "Custom Lyrics"])

    with overview_tab:
        render_analytics(df, model_name)
        st.subheader("Dataset Sources")
        source_counts = df.get("source", pd.Series(["dataset"] * len(df))).value_counts()
        st.dataframe(source_counts.rename("songs"), use_container_width=True)

    with catalog_tab:
        if search_query:
            song_options = recommender.search_songs(search_query, limit=80).index.tolist()
        else:
            song_options = df.index[:200].tolist()

        if not song_options:
            st.warning("No songs matched your search.")
        else:
            selected_index = st.selectbox(
                "Song",
                options=song_options,
                format_func=lambda index: format_song_option(df, index),
            )

            selected = df.loc[selected_index]
            render_selected_song(selected, recommender.lyrics_preview(selected))

            if st.button("Get Recommendations", type="primary", use_container_width=True):
                with st.spinner("Scoring semantic, emotion, theme, and genre signals..."):
                    recommendations = recommender.recommend_by_index(
                        int(selected_index),
                        top_n=top_n,
                        candidate_indices=candidate_indices,
                    )
                st.subheader("Recommendations")
                render_recommendations(recommendations)

    with custom_tab:
        custom_genre = st.selectbox(
            "Input genre",
            options=["metal", "rock", "hip-hop", "pop", "country", "hindi", "punjabi", "custom"],
            index=7,
        )
        custom_lyrics = st.text_area(
            "Lyrics",
            height=190,
            placeholder=(
                "Paste at least 20 words. Example: I carry blame through an empty street, "
                "sorry in my hands, regret in my chest, and every quiet night asks me to forgive myself..."
            ),
        )

        if st.button("Recommend From Custom Lyrics", type="primary", use_container_width=True):
            with st.spinner("Embedding custom lyrics and scoring recommendations..."):
                try:
                    query, recommendations = recommender.recommend_by_custom_lyrics(
                        custom_lyrics,
                        genre=custom_genre,
                        top_n=top_n,
                        candidate_indices=candidate_indices,
                    )
                except ValueError as exc:
                    st.warning(str(exc))
                else:
                    detected_cols = st.columns(3)
                    detected_cols[0].metric("Detected emotion", query.emotion)
                    detected_cols[1].metric("Detected theme", query.theme)
                    detected_cols[2].metric("Input genre", query.genre)
                    st.markdown(f"<p class='lyrics-preview'>{query.lyrics_preview}</p>", unsafe_allow_html=True)
                    render_recommendations(recommendations)


if __name__ == "__main__":
    main()


## Run The ML Pipeline In Notebook

In [ ]:
from data_loader import prepare_dataset, dataset_summary
from embedding import load_embedding_model, load_or_create_embeddings, selected_model_name
from recommender import MusicRecommender

df = prepare_dataset()
print(dataset_summary(df))
df[['song_name', 'artist', 'genre', 'emotion', 'theme', 'source']].head(10)


In [ ]:
model = load_embedding_model()
embeddings = load_or_create_embeddings(df, model, show_progress_bar=True)
print('Model:', selected_model_name(model))
print('Embeddings:', embeddings.shape)

recommender = MusicRecommender(df, embeddings, model)


In [ ]:
query_index = 0
query_song = df.loc[query_index, 'song_name']
print('Query:', query_song)

recommendations = recommender.recommend_by_index(query_index, top_n=5)
recommendations[[
    'song_name', 'artist', 'genre', 'emotion', 'theme',
    'semantic_similarity', 'final_score', 'explanation'
]]


In [ ]:
custom_lyrics = '''
I carry blame through an empty street and whisper sorry to the dark.
Regret keeps replaying inside my hollow chest while every mistake feels heavy.
The night feels sad and blue, and every mistake asks for forgiveness.
I feel alone, broken, and lost while searching for peace.
'''

query, recs = recommender.recommend_by_custom_lyrics(custom_lyrics, genre='metal', top_n=5)
print('Detected emotion:', query.emotion)
print('Detected theme:', query.theme)
recs[['song_name', 'artist', 'genre', 'emotion', 'theme', 'final_score', 'explanation']]


## Streamlit App

Run the UI from terminal:

```bash
streamlit run app.py
```